In [ ]:
# # from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.impute import SimpleImputer
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'sklearn'

In [26]:
df = pd.read_csv("PrepHealthDeficiencies_2019.csv", encoding='ISO-8859-1')

# Perform the counts calculation
provnum_counts = df['provnum'].value_counts().reset_index()

# Rename the columns to be clear
provnum_counts.columns = ['provnum', 'count']

print(provnum_counts)

# provnum_counts.to_csv('def2019.csv', index=False)

      provnum  count
0      505114    160
1      365206    159
2      145371    155
3      395382    154
4      145439    150
...       ...    ...
15320  265470      1
15321  676458      1
15322  345273      1
15323  445004      1
15324  415096      1

[15325 rows x 2 columns]


C:\Users\Julian Amberg\AppData\Local\Temp\ipykernel_9568\3337912503.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("PrepHealthDeficiencies_2019.csv", encoding='ISO-8859-1')


In [33]:
# Load the two CSV files
file1 = pd.read_csv('ProviderInfo1519_IQRCleaned.csv')
file2 = pd.read_csv('def2015.csv')

# Convert provnum to integers in both files
file1['provnum'] = file1['provnum'].astype(int)
file2['provnum'] = file2['provnum'].astype(int)

# Convert Year to string to ensure uniformity in data type
file1['Year'] = file1['Year'].astype(str).str.strip()
file2['Year'] = file2['Year'].astype(str).str.strip()

# Perform the merge
merged_df = pd.merge(file1, file2, on=['provnum', 'Year'], how='left')

# Check the result
print(merged_df[['provnum', 'Year', 'DeficiencyCount']].head())

merged_df.to_csv('merged_dfTest1.csv', index=False)


   provnum  Year  DeficiencyCount
0    15009  2015             17.0
1    15009  2016              NaN
2    15009  2017              NaN
3    15009  2018              NaN
4    15010  2015             25.0


In [38]:
# Load the next year's CSV (e.g., file3 for 2016)
file3 = pd.read_csv('def2016.csv')

# Clean and process the next year's file (file3)
# Remove rows with NaN in provnum and convert to string
file3 = file3.dropna(subset=['provnum'])
file3['provnum'] = file3['provnum'].astype(str)
file3['Year'] = file3['Year'].astype(str).str.strip()

# Filter rows where provnum is numeric
file3 = file3[file3['provnum'].str.isnumeric()]

# Convert provnum to integers
file3['provnum'] = file3['provnum'].astype(int)

# Perform the left merge (this will keep all rows from merged_df)
merged_df = pd.merge(merged_df, file3[['provnum', 'Year', 'DeficiencyCount']], on=['provnum', 'Year'], how='left', suffixes=('', '_new'))

# Now, fill in the missing values in 'DeficiencyCount' with values from 'DeficiencyCount_new' where applicable
merged_df['DeficiencyCount'] = merged_df['DeficiencyCount'].fillna(merged_df['DeficiencyCount_new'])

# Drop the temporary column 'DeficiencyCount_new' to clean up
merged_df = merged_df.drop(columns=['DeficiencyCount_new'])

# Check the result
print(merged_df[['provnum', 'Year', 'DeficiencyCount']].head())


   provnum  Year  DeficiencyCount
0    15009  2015             17.0
1    15009  2016              7.0
2    15009  2017              NaN
3    15009  2018              NaN
4    15010  2015             25.0


In [ ]:
# Ensure merged_df provnum and Year are correct BEFORE the loop
merged_df['provnum'] = merged_df['provnum'].astype(str)
merged_df['Year'] = merged_df['Year'].astype(str).str.strip()

# List of CSV files to merge
files = ['def2017.csv', 'def2018.csv', 'def2019.csv']

for file in files:
    new_data = pd.read_csv(file)

    # Drop rows missing provnum
    new_data = new_data.dropna(subset=['provnum'])

    # Force numeric provnum and Year
    new_data['provnum'] = pd.to_numeric(new_data['provnum'], errors='coerce')
    new_data['Year'] = pd.to_numeric(new_data['Year'], errors='coerce')

    # Drop any that failed to convert
    new_data = new_data.dropna(subset=['provnum', 'Year'])
    new_data['provnum'] = new_data['provnum'].astype(int)
    new_data['Year'] = new_data['Year'].astype(int)

    # Ensure merged_df is also clean
    merged_df['provnum'] = pd.to_numeric(merged_df['provnum'], errors='coerce')
    merged_df['Year'] = pd.to_numeric(merged_df['Year'], errors='coerce')
    merged_df = merged_df.dropna(subset=['provnum', 'Year'])
    merged_df['provnum'] = merged_df['provnum'].astype(int)
    merged_df['Year'] = merged_df['Year'].astype(int)

    # Debug
    print("Matching provnums:", set(merged_df['provnum']).intersection(set(new_data['provnum'])))
    print("Matching Years:", set(merged_df['Year']).intersection(set(new_data['Year'])))

    # Merge
    merged_df = pd.merge(
        merged_df, 
        new_data[['provnum', 'Year', 'DeficiencyCount']], 
        on=['provnum', 'Year'], 
        how='left', 
        suffixes=('', '_new')
    )

    # Fill missing
    merged_df['DeficiencyCount'] = merged_df['DeficiencyCount'].fillna(merged_df['DeficiencyCount_new'])
    merged_df = merged_df.drop(columns=['DeficiencyCount_new'])

print(merged_df[['provnum', 'Year', 'DeficiencyCount']].head())

merged_df.to_csv('merged_PrDe_data', index=False)

Matching provnums: {295000, 295001, 295006, 295008, 295011, 295017, 295020, 295021, 295029, 295036, 295040, 295043, 295044, 295045, 295046, 295055, 295066, 295067, 295068, 295070, 295072, 295075, 295077, 295078, 295079, 295080, 295081, 295082, 295086, 295088, 295089, 295093, 295094, 495109, 525009, 525019, 525064, 525074, 525085, 525088, 525098, 525132, 525165, 525172, 525179, 525209, 525232, 525262, 525264, 525265, 525270, 525271, 525274, 525276, 525286, 525292, 525299, 525304, 525305, 525306, 525314, 525315, 525317, 525318, 525319, 525321, 525324, 525326, 525327, 525328, 525329, 525331, 525332, 525334, 525335, 525336, 525337, 525338, 525339, 495188, 525342, 525343, 525346, 525348, 525350, 525351, 525352, 525353, 525354, 525355, 525357, 525358, 525359, 525360, 525362, 525363, 525365, 525369, 525370, 525372, 525373, 525375, 525376, 525380, 525383, 525389, 495199, 525395, 525396, 495200, 525398, 525402, 525403, 525406, 525407, 525408, 525409, 525410, 525411, 525412, 525413, 525414, 5254